<a href="https://www.kaggle.com/code/asopozala/bbc-sentiment-classifier?scriptVersionId=320116676" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

I experiment with building a sentiment classifier for BBC‑style text, using engineered features from news articles. The notebook walks through feature extraction, model training, evaluation metrics, and a discussion of where the model succeeds and fails.

#  What I learned from a “bad outcome”
This notebook taught me that strong metrics can be misleading when my labels come from a proxy labeler (VADER).  
The TF‑IDF + Logistic Regression model achieved high accuracy, but mainly because it learned to **imitate VADER’s lexical rule**, not because it discovered human-level sentiment reasoning.  
So the outcome is useful: it clarifies the difference between *learning a real concept* vs *learning the labeling function*.


In [1]:
import pandas as pd

data_path = "/kaggle/input/datasets/gpreda/bbc-news/bbc_news.csv"  # same as before
df = pd.read_csv(data_path)

df["pubDate_dt"] = pd.to_datetime(df["pubDate"], errors="coerce")
df["year"] = df["pubDate_dt"].dt.year
df["text"] = df["title"].astype(str) + " " + df["description"].astype(str)

df[["year", "title", "description"]].head()


,year,title,description
0,2022,Ukraine: Angry Zelensky vows to punish Russian...,The Ukrainian president says the country will ...
1,2022,War in Ukraine: Taking cover in a town under a...,"Jeremy Bowen was on the frontline in Irpin, as..."
2,2022,Ukraine war 'catastrophic for global food',One of the world's biggest fertiliser firms sa...
3,2022,Manchester Arena bombing: Saffie Roussos's par...,The parents of the Manchester Arena bombing's ...
4,2022,Ukraine conflict: Oil price soars to highest l...,Consumers are feeling the impact of higher ene...


In [2]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# download lexicon once (safe to run multiple times)
nltk.download("vader_lexicon")

sia = SentimentIntensityAnalyzer()

def get_compound(text):
    return sia.polarity_scores(text)["compound"]

df["sentiment"] = df["text"].apply(get_compound)

df[["year", "title", "sentiment"]].head()


[nltk_data] Error loading vader_lexicon: <urlopen error [Errno -3]
[nltk_data]     Temporary failure in name resolution>


,year,title,sentiment
0,2022,Ukraine: Angry Zelensky vows to punish Russian...,-0.9109
1,2022,War in Ukraine: Taking cover in a town under a...,-0.8555
2,2022,Ukraine war 'catastrophic for global food',-0.9001
3,2022,Manchester Arena bombing: Saffie Roussos's par...,-0.5267
4,2022,Ukraine conflict: Oil price soars to highest l...,0.0772


In [3]:
import numpy as np

def label_sentiment(c):
    if c <= -0.2:
        return "negative"
    elif c >= 0.2:
        return "positive"
    else:
        return "neutral"

df["sentiment_label"] = df["sentiment"].apply(label_sentiment)

print(df["sentiment_label"].value_counts())
df[["title", "sentiment", "sentiment_label"]].head(10)


sentiment_label
negative    18270
positive    13749
neutral     10096
Name: count, dtype: int64


,title,sentiment,sentiment_label
0,Ukraine: Angry Zelensky vows to punish Russian...,-0.9109,negative
1,War in Ukraine: Taking cover in a town under a...,-0.8555,negative
2,Ukraine war 'catastrophic for global food',-0.9001,negative
3,Manchester Arena bombing: Saffie Roussos's par...,-0.5267,negative
4,Ukraine conflict: Oil price soars to highest l...,0.0772,neutral
5,Ukraine war: PM to hold talks with world leade...,-0.5994,negative
6,Ukraine war: UK grants 50 Ukrainian refugee vi...,-0.2500,negative
7,TikTok limits services as Netflix pulls out of...,-0.2960,negative
8,"Covid: Fourth jab for Scotland's vulnerable, a...",-0.5719,negative
9,Protests across Russia see thousands detained,-0.5574,negative


In [4]:
cols = ["year", "title", "sentiment", "sentiment_label", "link"]

sample = df[cols].sample(6, random_state=42)
sample


,year,title,sentiment,sentiment_label,link
24790,2023,US's only Palestinian-American Congresswoman c...,0.0000,neutral,https://www.bbc.co.uk/news/world-us-canada-673...
29911,2024,Murdered driver's family demand help for couriers,-0.9081,negative,https://www.bbc.co.uk/news/uk-wales-67726081
21220,2023,Cleared pony owner criticises 'trial by social...,-0.7096,negative,https://www.bbc.co.uk/news/uk-england-leiceste...
16716,2023,Wrexham: Can football fame make City of Cultur...,0.9231,positive,https://www.bbc.co.uk/news/uk-wales-65494230?a...
32850,2024,Gaza ceasefire talks intensify in Cairo,0.6486,positive,https://www.bbc.co.uk/news/world-middle-east-6...
20177,2023,Bud Light boycott over trans influencer Dylan ...,0.4939,positive,https://www.bbc.co.uk/news/business-66398296?a...


In [5]:
# Option 1: show full links for your sample rows
sample["link"].tolist()



['https://www.bbc.co.uk/news/world-us-canada-67354706?at_medium=RSS&at_campaign=KARANGA',
 'https://www.bbc.co.uk/news/uk-wales-67726081',
 'https://www.bbc.co.uk/news/uk-england-leicestershire-66605870?at_medium=RSS&at_campaign=KARANGA',
 'https://www.bbc.co.uk/news/uk-wales-65494230?at_medium=RSS&at_campaign=KARANGA',
 'https://www.bbc.co.uk/news/world-middle-east-68956549',
 'https://www.bbc.co.uk/news/business-66398296?at_medium=RSS&at_campaign=KARANGA']

In [6]:
def label_sentiment(c):
    if c <= -0.4:
        return "negative"
    elif c >= 0.4:
        return "positive"
    else:
        return "neutral"

df["sentiment_label"] = df["sentiment"].apply(label_sentiment)

cols = ["year", "title", "sentiment", "sentiment_label", "link"]
df[cols].sample(6)   # re‑run this cell to see a new random set each time


,year,title,sentiment,sentiment_label,link
31625,2024,Biden vows to help Baltimore recover rapidly,0.1531,neutral,https://www.bbc.co.uk/news/world-us-canada-687...
19140,2023,Ukraine war: The lethal minefields holding up ...,-0.5994,negative,https://www.bbc.co.uk/news/world-europe-660806...
6352,2022,Is the UK heading for a drought and will there...,-0.4767,negative,https://www.bbc.co.uk/news/science-environment...
13549,2023,A secret room that saved this girl's life,0.8225,positive,https://www.bbc.co.uk/news/world-europe-645133...
20535,2023,Mortgage rates: Five ways to save money,0.2500,neutral,https://www.bbc.co.uk/news/business-65984415?a...
23870,2023,"Man Utd wanted to win for legend Charlton, say...",0.9538,positive,https://www.bbc.co.uk/sport/football/67109090?...


In [7]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

idx = [12177, 36877, 40750, 26414, 39086, 6136]

sample = df.loc[idx, ["year", "title", "sentiment", "sentiment_label", "link"]]
print(sample, "\n")

print("Links only:")
print(sample["link"].tolist())


       year  \
12177  2022   
36877  2024   
40750  2024   
26414  2023   
39086  2024   
6136   2022   

                                                                           title  \
12177               Energy giant ExxonMobil sues EU to block energy windfall tax   
36877                          Are the UK's finances really worse than expected?   
40750                                                      Tommy Robinson Jailed   
26414               Why Wagner is winning hearts in the Central African Republic   
39086                        SpaceX crew returns to Earth after historic mission   
6136   England win Euro 2022: Lionesses fans revel in final victory over Germany   

       sentiment sentiment_label  \
12177     0.1027         neutral   
36877    -0.5256        negative   
40750    -0.4939        negative   
26414     0.7915        positive   
39086     0.0000         neutral   
6136      0.9153        positive   

                                                    

## Sentiment labels: VADER vs Gemini

We used VADER (`SentimentIntensityAnalyzer`) to assign each BBC article a sentiment `compound` score in [-1, 1], then mapped it to three labels using thresholds at -0.4 and +0.4.  
After sampling 6 random articles and comparing with Gemini’s human‑style judgment, we saw that only 2/6 matched clearly; 2 were mildly different (borderline cases), and 2 were strongly different on complex, mixed articles.  
This means our current labels are **noisy**, because we trust VADER even when the score is not very extreme.  
To get a cleaner supervised task for scikit‑learn, we will now **tighten the thresholds** (e.g. `compound <= -0.6` → negative, `>= +0.6` → positive, everything else neutral/ignored), trading quantity of data for higher label quality.  
Then we will train a `TfidfVectorizer + LogisticRegression` classifier only on clearly positive/negative examples and evaluate how well it learns these strong sentiments.


In [8]:
def label_strong_sentiment(c):
    if c <= -0.6:
        return "negative"
    elif c >= 0.6:
        return "positive"
    else:
        return "neutral"

df["sentiment_label"] = df["sentiment"].apply(label_strong_sentiment)

print("Label counts (strong thresholds ±0.6):")
print(df["sentiment_label"].value_counts(), "\n")

# Keep only clear positive/negative for training
mask = df["sentiment_label"].isin(["positive", "negative"])
train_df = df[mask].copy()

print("Training set size:", len(train_df))
train_df[["title", "sentiment", "sentiment_label"]].head(6)


Label counts (strong thresholds ±0.6):
sentiment_label
neutral     25584
negative    10078
positive     6453
Name: count, dtype: int64 

Training set size: 16531


,title,sentiment,sentiment_label
0,Ukraine: Angry Zelensky vows to punish Russian atrocities,-0.9109,negative
1,War in Ukraine: Taking cover in a town under attack,-0.8555,negative
2,Ukraine war 'catastrophic for global food',-0.9001,negative
10,Ukraine conflict: Your guide to understanding day 11,-0.7351,negative
12,"Ukraine: With placards and tears, Poles are greeting refugees like family",0.7351,positive
15,Ukraine crisis: The West fights back against Putin the disruptor,-0.6757,negative


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Use only clear positive/negative examples
X = train_df["text"]
y = train_df["sentiment_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", max_df=0.8, min_df=10)),
    ("logreg", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

    negative       0.95      0.97      0.96      2016
    positive       0.95      0.91      0.93      1291

    accuracy                           0.95      3307
   macro avg       0.95      0.94      0.95      3307
weighted avg       0.95      0.95      0.95      3307

[[1960   56]
 [ 112 1179]]


In [10]:
new_articles = [
    """Cuba has completely run out of diesel and fuel oil, the country's Energy Minister Vicente de la O Levy has said.

In an interview with state-run media, de la O Levy said there were limited amounts of gas available, but that Cuba's energy system was in a "critical" state as a US-led blockade of oil to the country squeezed supply.

Scattered protests against power cuts broke out in the capital Havana on Wednesday, the Reuters news agency reported.

The US this week reiterated its offer of sending $100m (£74m) in aid to the country in exchange for "meaningful reforms to Cuba's communist system".

"The sum of the different types of fuel: crude oil, fuel oil, of which we have absolutely none; diesel, of which we have absolutely none - I am being repetitive - the only thing we have is gas from our wells, where production has grown," de la O Levy said.

Under the US blockade, parts of Havana have been plunged into 20 to 22-hour blackout periods, he continued.

He also acknowledged that the situation in the country had been "extremely tense".

Hospitals have been unable to function normally, while schools and government offices have been forced to close. Tourism, an economic engine for Cuba, has also been impacted.
""",
    """'They shot my neighbour in the head' - the lakeside city traumatised by war

Summary executions and rape were among the atrocities committed by the M23 rebel group and Rwandan soldiers during their weeks-long occupation of the lakeside city of Uvira in eastern Democratic Republic of Congo, an investigation by a leading rights group has found.

Human Rights Watch (HRW) says its investigators found evidence of the execution of 53 civilians - 46 men, one woman and six children - during door-to-door raids in the city's neighbourhoods after the rebels, widely believed to be backed by Rwanda, captured it in December.

Rwanda has consistently denied supporting the M23 or that its own soldiers have been deployed to resource-rich eastern DR Congo.

But HRW says many of the interviewees alleged witnessing atrocities committed by uniformed Rwandan soldiers as well as M23 fighters.

"They [M23 fighters] shot my neighbour first in the head," said one of the 130 residents interviewed by HRW.

Another said he saw four members of his family killed.

"I wasn't hit so I just ran to the lake. I saw my brother, his wife, and two of his children fall," he was quoted as saying.

The M23 and Rwandan government have not yet responded to a BBC request for comment.
"""
]

pred_labels = model.predict(new_articles)
pred_probs = model.predict_proba(new_articles)

for i, (text, label, probs) in enumerate(zip(new_articles, pred_labels, pred_probs), start=1):
    print(f"ARTICLE {i}")
    print("Predicted label:", label)
    print("Prob(negative), Prob(positive):", probs)
    print("Preview:", text[:180].replace("\n", " "), "...")
    print("-" * 80)



ARTICLE 1
Predicted label: negative
Prob(negative), Prob(positive): [0.88738616 0.11261384]
Preview: Cuba has completely run out of diesel and fuel oil, the country's Energy Minister Vicente de la O Levy has said.  In an interview with state-run media, de la O Levy said there were ...
--------------------------------------------------------------------------------
ARTICLE 2
Predicted label: negative
Prob(negative), Prob(positive): [0.95931789 0.04068211]
Preview: 'They shot my neighbour in the head' - the lakeside city traumatised by war  Summary executions and rape were among the atrocities committed by the M23 rebel group and Rwandan sold ...
--------------------------------------------------------------------------------


### Summary so far – BBC sentiment classifier

- I used VADER (via NLTK) to assign each BBC article a `compound` sentiment score in [-1, 1].
- To reduce noise, I kept only **strong** scores (`compound <= -0.6` as negative, `>= 0.6` as positive) and trained a `TfidfVectorizer + LogisticRegression` classifier on those clear examples.
- The classifier reaches about **95% accuracy** on a held-out test set, so it has effectively learned a VADER‑style sentiment rule directly from text using scikit‑learn.
- When I test it on new BBC articles from this week, it confidently predicts **negative** sentiment for crisis/war stories, which matches my own intuition but sometimes disagrees with Gemini’s more nuanced judgment.
- This shows the difference between a **lexical pattern model** (my classifier) and a **reasoning model** (Gemini), and gives me a concrete tool I can now use to estimate sentiment for new news articles.
